# Machine Condition Classification — Strictly Bayes Classifiers
Only generative Bayes classifiers (class prior x class-conditional density, then Bayes' rule): Gaussian Naive Bayes, LDA, QDA and GMM-Bayes, with 5-fold stratified CV.
Logistic regression (MAP) and blends that include it have been removed.

In [ ]:
import numpy as np
import pandas as pd
import warnings
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA, QuadraticDiscriminantAnalysis as QDA
from sklearn.mixture import GaussianMixture
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")

In [ ]:
# Config and paths
DATA_DIR = "."
TRAIN_FILE = "student_dataset.csv"
TEST_FILE = "test.csv"
SEED = 42

train = pd.read_csv(f"{DATA_DIR}/{TRAIN_FILE}")
test = pd.read_csv(f"{DATA_DIR}/{TEST_FILE}")
y = train["condition"].values

NUM = [
    "ambient_temp", "process_temp", "coolant_flow", "shaft_speed",
    "drive_torque", "power_load", "component_wear", "thermal_stress",
    "mechanical_load", "vibration_index"
]

print(train.shape, test.shape, "| class share:", np.bincount(y) / len(y))
print("missing cells train/test:", int(train[NUM].isna().sum().sum()), int(test[NUM].isna().sum().sum()))

### Feature Engineering & Preprocessing
Missing values are imputed inside each fold (Bayesian-ridge iterative imputer) so no validation data leaks, then domain ratios and log transforms are added.

In [ ]:
MODE_TYPE = train["unit_type"].mode()[0]

def engineer(d):
    X = d[NUM].copy()
    ut = d["unit_type"].fillna(MODE_TYPE)
    for t in "SMH":
        X["T_" + t] = (ut == t).astype(float)

    # interaction terms
    X["dT"] = X.process_temp - X.ambient_temp
    X["tor_wear"] = X.drive_torque * X.component_wear
    X["ts_flow"] = X.thermal_stress / X.coolant_flow
    X["mech_flow"] = X.mechanical_load / X.coolant_flow
    X["wear_mech"] = X.component_wear * X.mechanical_load
    X["vib_speed"] = X.vibration_index / X.shaft_speed * 1000

    # log transform right-skewed features
    skewed_cols = [
        "component_wear", "vibration_index", "power_load", "thermal_stress",
        "mechanical_load", "coolant_flow", "tor_wear", "ts_flow",
        "mech_flow", "wear_mech", "shaft_speed", "drive_torque"
    ]
    for c in skewed_cols:
        X["log_" + c] = np.log(X[c].clip(lower=1e-3))
    return X

In [ ]:
class Prep:
    # Fit imputer and scalers on train split only
    def __init__(self, cols_kind="eng", degree=1):
        self.cols_kind = cols_kind
        self.degree = degree

    def _cols(self, X):
        base = NUM + ["T_S", "T_M", "T_H"]
        return base if self.cols_kind == "raw" else list(X.columns)

    def _impute_eng(self, d, fit):
        D = d.copy()
        if fit:
            self.imp = IterativeImputer(estimator=BayesianRidge(), max_iter=8, random_state=SEED).fit(D[NUM])
        D[NUM] = self.imp.transform(D[NUM])
        X = engineer(D)
        return X[self._cols(X)]

    def fit_transform(self, d):
        X = self._impute_eng(d, True)
        self.s1 = StandardScaler().fit(X)
        Z = self.s1.transform(X)
        if self.degree > 1:
            self.pf = PolynomialFeatures(self.degree, include_bias=False)
            Z = self.pf.fit_transform(Z)
            self.s2 = StandardScaler().fit(Z)
            Z = self.s2.transform(Z)
        return Z

    def transform(self, d):
        Z = self.s1.transform(self._impute_eng(d, False))
        if self.degree > 1:
            Z = self.s2.transform(self.pf.transform(Z))
        return Z

### Custom GMM Bayes & CV Harness

In [ ]:
# Fit a separate Gaussian Mixture per class and weight by class prior (Bayes' rule with mixture densities)
class GMMBayes:
    def __init__(self, k=2, reg=1e-2):
        self.k = k
        self.reg = reg

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.logprior = np.log([(y == c).mean() for c in self.classes_])
        self.gm = [
            GaussianMixture(self.k, covariance_type="full", reg_covar=self.reg, random_state=SEED).fit(X[y == c])
            for c in self.classes_
        ]
        return self

    def predict_proba(self, X):
        L = np.column_stack([g.score_samples(X) for g in self.gm]) + self.logprior
        L -= L.max(1, keepdims=True)
        P = np.exp(L)
        return P / P.sum(1, keepdims=True)

In [ ]:
skf = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(train, y))

# Cache the imputed/scaled folds since IterativeImputer takes a few seconds
FOLD_CACHE = {}

def folds(kind, degree):
    key = (kind, degree)
    if key not in FOLD_CACHE:
        out = []
        for a, b in skf:
            p = Prep(kind, degree)
            Xa = p.fit_transform(train.iloc[a])
            Xb = p.transform(train.iloc[b])
            out.append((Xa, Xb))
        FOLD_CACHE[key] = out
    return FOLD_CACHE[key]

OOF, RESULTS, SPECS = {}, [], {}   # SPECS remembers how to rebuild each model for the final fit

def evaluate(name, kind, degree, make_model):
    t = time.time()
    SPECS[name] = (kind, degree, make_model)
    P = np.zeros((len(y), 3))
    accs = []
    for (a, b), (Xa, Xb) in zip(skf, folds(kind, degree)):
        m = make_model().fit(Xa, y[a])
        P[b] = m.predict_proba(Xb)
        accs.append((P[b].argmax(1) == y[b]).mean())
    OOF[name] = P
    RESULTS.append({"model": name, "cv_acc": np.mean(accs), "std": np.std(accs)})
    print(f"{name:42s} acc={np.mean(accs):.4f} +/- {np.std(accs):.4f}  ({time.time()-t:.0f}s)", flush=True)

### Running the Models (strictly Bayes only)

In [ ]:
# Baselines on raw features
evaluate("Majority baseline (class prior only)", "raw", 1, lambda: DummyClassifier(strategy="prior"))
evaluate("Gaussian Naive Bayes (raw)", "raw", 1, GaussianNB)
evaluate("LDA (raw)", "raw", 1, LDA)
evaluate("QDA reg=0.1 (raw)", "raw", 1, lambda: QDA(reg_param=0.1))
evaluate("GMM-Bayes k=2 (raw)", "raw", 1, lambda: GMMBayes(2, 1e-2))

In [ ]:
# Engineered features and degree-2 polynomial expansions (still LDA = Gaussian class-conditionals, shared covariance)
evaluate("LDA (engineered)", "eng", 1, LDA)
evaluate("LDA quadratic features (raw)", "raw", 2, LDA)
evaluate("LDA quadratic features (eng, shrinkage)", "eng", 2, lambda: LDA(solver="lsqr", shrinkage="auto"))

### Results Comparison

In [ ]:
table = pd.DataFrame(RESULTS).sort_values("cv_acc", ascending=False)
table.to_csv("model_comparison_bayes.csv", index=False)
print(table.to_string(index=False))

winner = table.iloc[0].model
print("\nWinner:", winner)
print(confusion_matrix(y, OOF[winner].argmax(1)))

### Final Training & Submission (best strictly Bayes model)

In [ ]:
# Fit the winning model on the full training set and generate test predictions
kind, degree, make_model = SPECS[winner]
p = Prep(kind, degree)
Xa = p.fit_transform(train)
Xt = p.transform(test)
P_final = make_model().fit(Xa, y).predict_proba(Xt)

sub = pd.DataFrame({
    "unit_id": test["unit_id"].values,
    "condition": P_final.argmax(1).astype(int)
})
sub.to_csv("submission_bayes.csv", index=False)

print("submission:", sub.shape, "| class share:", np.bincount(sub.condition, minlength=3) / len(sub))
sub.head()